# **modelleri hugging face hub'a aktarma**

In [1]:
from google.colab import drive
# 1. DRIVE BAĞLANTISI VE VERİ ÇIKARMA
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Önce Drive'dan Colab'ın içine kopyala
!cp /content/drive/MyDrive/akilliTarimOdevi/v3-801010/tinyllama_output_zip.zip /content/
!cp /content/drive/MyDrive/akilliTarimOdevi/v3-801010/gemma4_output_zip.zip /content/
!cp /content/drive/MyDrive/akilliTarimOdevi/v3-801010/smollm2_output_zip.zip /content/
# Sonra zip'i aç
!unzip -q /content/drive/MyDrive/akilliTarimOdevi/v3-801010/tinyllama_output_zip.zip -d /content/tinyllama_output_zip
!unzip -q /content/drive/MyDrive/akilliTarimOdevi/v3-801010/gemma4_output_zip.zip -d /content/gemma4_output_zip
!unzip -q /content/drive/MyDrive/akilliTarimOdevi/v3-801010/smollm2_output_zip.zip -d /content/smollm2_output_zip

In [3]:
# Önce Drive'dan Colab'ın içine kopyala
!cp /content/drive/MyDrive/akilliTarimOdevi/v3-801010/qwen_output_zip.zip /content/
# Sonra zip'i aç
!unzip -q /content/drive/MyDrive/akilliTarimOdevi/v3-801010/qwen_output_zip.zip -d /content/qwen_output_zip

In [3]:
!pip install -q "torchao>=0.16.0" --upgrade
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **en başarılı model**

In [1]:
from huggingface_hub import login
from google.colab import userdata
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc

login(token=userdata.get("HF_TOKEN"))

print("⏳ Qwen yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.bfloat16,      # torch_dtype yerine dtype
    device_map="auto"
)
model = PeftModel.from_pretrained(
    model, "/content/qwen_output_zip/seed_42/best_model"
)

tokenizer.push_to_hub("haticenuryavas/qwen2.5-1.5b-tarim-lora-seed42")
model.push_to_hub("haticenuryavas/qwen2.5-1.5b-tarim-lora-seed42")

del model, tokenizer
gc.collect(); torch.cuda.empty_cache()
print("✅ Qwen yüklendi!")

⏳ Qwen yükleniyor...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpt_c8fxdm/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

✅ Qwen yüklendi!


# **diğer modeller**

In [6]:
from huggingface_hub import login
from google.colab import userdata
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc

login(token=userdata.get("HF_TOKEN"))

MODELLER = [
    {
        "base":    "HuggingFaceTB/SmolLM2-360M-Instruct",
        "adapter": "/content/smollm2_output_zip/seed_{}/best_model",
        "hub":     "haticenuryavas/smollm2-360m-tarim-lora-seed{}",
    },
    {
        "base":    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "adapter": "/content/tinyllama_output_zip/seed_{}/best_model",
        "hub":     "haticenuryavas/tinyllama-1.1b-tarim-lora-seed{}",
    },
    {
        "base":    "google/gemma-4-E2B-it",
        "adapter": "/content/gemma4_output_zip/seed_{}/best_model",
        "hub":     "haticenuryavas/gemma4-e2b-tarim-lora-seed{}",
    },
    # Qwen sadece seed 123 ve 7 — 42 zaten var
    {
        "base":    "Qwen/Qwen2.5-1.5B-Instruct",
        "adapter": "/content/qwen_output_zip/seed_{}/best_model",
        "hub":     "haticenuryavas/qwen2.5-1.5b-tarim-lora-seed{}",
        "seeds":   [123, 7],  # 42 atla
    },
]

for m in MODELLER:
    seeds = m.get("seeds", [42, 123, 7])
    for seed in seeds:
        print(f"\n⏳ {m['hub'].format(seed)} yükleniyor...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(m["base"])
            model = AutoModelForCausalLM.from_pretrained(
                m["base"], dtype=torch.bfloat16, device_map="auto"
            )
            model = PeftModel.from_pretrained(
                model, m["adapter"].format(seed)
            )
            tokenizer.push_to_hub(m["hub"].format(seed))
            model.push_to_hub(m["hub"].format(seed))
            print(f"✅ {m['hub'].format(seed)} tamamlandı!")
        except Exception as e:
            print(f"❌ Hata: {e}")
        finally:
            del model, tokenizer
            gc.collect(); torch.cuda.empty_cache()

print("\n🎉 Tüm modeller yüklendi!")


⏳ haticenuryavas/smollm2-360m-tarim-lora-seed42 yükleniyor...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

✅ haticenuryavas/tinyllama-1.1b-tarim-lora-seed7 tamamlandı!


You are using a model of type gemma4 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.



⏳ haticenuryavas/gemma4-e2b-tarim-lora-seed42 yükleniyor...
❌ Hata: The checkpoint you are trying to load has model type `gemma4` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`


NameError: name 'model' is not defined

In [2]:
!pip install -q git+https://github.com/huggingface/transformers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# **gemma4 adapter yüklemesi **

In [3]:
from huggingface_hub import HfApi, login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))
api = HfApi()

for seed in [42, 123, 7]:
    repo_id = f"haticenuryavas/gemma4-e2b-tarim-lora-seed{seed}"
    api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")
    api.upload_folder(
        folder_path=f"/content/gemma4_output_zip/seed_{seed}/best_model",
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"✅ Gemma4 seed {seed} yüklendi!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         | 3.28MB /  152MB            

  ...best_model/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

  ...t_model/training_args.bin:  97%|#########7| 5.51kB / 5.65kB            

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         | 3.28MB /  152MB            

  ...best_model/tokenizer.json: 100%|##########| 32.2MB / 32.2MB            

  ...t_model/training_args.bin: 100%|##########| 5.65kB / 5.65kB            

✅ Gemma4 seed 7 yüklendi!
